# R2-10 · Series temporales — Lección

**Bootcamp de Datos para Funcionarios Públicos · Formación Pública**
Línea R2 · *Científico de Datos* · Semana 16 (optativa) · Se ejecuta en Google Colab.

> 🚀 **Cómo se trabaja:** lee, ejecuta cada celda con **▶︎** (o `Shift`+`Enter`) y completa los `TODO`. Cada ejercicio termina en una **celda de chequeo** que muestra ✅ si está bien o una pista si todavía no.

---

## ¿Qué vas a lograr?

1. Construir una **serie diaria** con calendario completo y contar sus huecos.
2. **Remuestrear** (semana, mes) y suavizar con **media móvil**.
3. Medir la **estacionalidad** anual y semanal y **descomponer** la serie.
4. Comparar pronósticos con **validación temporal** y probarlos contra **2026**.

**Competencia de salida:** dado un registro temporal real, describir su forma y producir un pronóstico validado que le gane (o no) a la línea base, y decirlo con honestidad.

**Dato real:** PM2.5 diario de la estación Cerro Navia (SINCA, 2020–2025) y lo que pasó en enero–septiembre de 2026.

**Entregable:** que las **7 celdas de chequeo** muestren ✅.

In [ ]:
# ── Preparación del entorno (ejecuta esta celda primero) ──────────────────────
import os, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# "En vivo o caché": usa los archivos locales; si faltan (ej. Colab), se bajan del repo.
for archivo in ["aire_diario.csv", "aire_nuevos.csv"]:
    if not os.path.exists(archivo):
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/data/{archivo}", archivo)

aire = pd.read_csv("aire_diario.csv", parse_dates=["fecha"])     # 2020–2025, 14 estaciones
nuevos = pd.read_csv("aire_nuevos.csv", parse_dates=["fecha"])   # ene–sep 2026: el futuro "real"
ESTACION = "Cerro Navia"                                          # comuna de Cerro Navia, Región Metropolitana
print("Datos:", aire.shape, "| estaciones:", aire["estacion"].nunique())
aire[aire["estacion"] == ESTACION].head(3)

## ¿Qué es una serie temporal?

Una **serie temporal** es una secuencia de valores **ordenados en el tiempo** y medidos a
intervalos regulares: el PM2.5 de cada día, las solicitudes de cada semana, las atenciones de
cada mes. La palabra clave es *ordenados*: si desordenas las filas, pierdes lo más valioso — si
venía subiendo, si cada invierno se dispara, hacia dónde apunta. **La fila de al lado importa.**

Casi cualquier serie se puede pensar como la suma de tres ingredientes:

1. **Tendencia:** hacia dónde va a largo plazo (¿el aire de Santiago mejora año a año?).
2. **Estacionalidad:** patrones que se **repiten** en ciclos fijos. En el PM2.5 de Santiago, el
   invierno es inconfundible: frío, poca ventilación y calefacción. Con datos **diarios** también
   podemos buscar un ciclo **semanal**.
3. **Ruido:** lo que queda. Ningún pronóstico lo elimina; un buen análisis no lo confunde con señal.

Hoy trabajamos con el PM2.5 diario de la estación **Cerro Navia** (2020–2025) y, al final, lo
ponemos a prueba contra lo que **de verdad pasó** en 2026.

> Norma chilena de PM2.5 para 24 horas: **50 µg/m³** (D.S. 12/2011 del Ministerio del Medio Ambiente).

## 1. Construir la serie: el tiempo como índice

El primer paso siempre es el mismo: quedarte con una estación, dejar la **fecha como índice** y exigir un **calendario completo** con `.asfreq("D")`. Si falta un día, debe aparecer como `NaN` — no desaparecer. Una serie con días «saltados» engaña a todo lo que viene después (medias móviles, desfases, pronósticos).

### ✍️ Ejercicio 1 — Arma la serie diaria y cuenta los días sin dato

In [ ]:
# TODO: filtra ESTACION, deja "fecha" como índice, toma "pm25", ordena y fuerza frecuencia diaria
pm = ...
n_dias = ...        # TODO: largo de la serie
n_faltantes = ...   # TODO: días con pm25 nulo
print("días:", n_dias, "| sin dato:", n_faltantes)

In [ ]:
# ── Celda de chequeo · Ejercicio 1 ──────────────────────────────────────────
try:
    assert isinstance(pm, pd.Series) and isinstance(pm.index, pd.DatetimeIndex)
    assert pm.index.freqstr == "D", "usa .asfreq('D') para tener un calendario completo"
    assert n_dias == len(pm) == (pm.index.max() - pm.index.min()).days + 1
    assert n_faltantes == int(pm.isna().sum()) and n_faltantes > 0
    print("✅ Ejercicio 1: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'aire[aire["estacion"] == ESTACION].set_index("fecha")["pm25"].sort_index().asfreq("D"); len(pm) y pm.isna().sum().')
    print("   Detalle:", e)

## 2. Cambiar la escala del tiempo: remuestreo y media móvil

El dato diario es ruidoso. Dos herramientas para ver la forma:

- **Remuestrear** (`resample`): pasar de días a semanas (`"W"`) o meses (`"MS"`, inicio de mes), promediando. Es la tabla dinámica del tiempo.
- **Media móvil** (`rolling`): el promedio de los últimos 30 días, recalculado cada día. Suaviza sin cambiar la frecuencia. Con `min_periods=20` exigimos al menos 20 días con dato.

### ✍️ Ejercicio 2 — Serie semanal, mensual y media móvil de 30 días

In [ ]:
# TODO: promedio semanal ("W") y mensual ("MS") con resample
semanal = ...
mensual = ...
# TODO: media móvil de 30 días con al menos 20 días con dato
movil_30 = ...
print("semanas:", len(semanal) if hasattr(semanal, "__len__") else semanal)

In [ ]:
# ── Celda de chequeo · Ejercicio 2 ──────────────────────────────────────────
try:
    assert len(mensual) == 72 and abs(mensual.iloc[0] - pm["2020-01"].mean()) < 1e-9
    assert len(semanal) == len(pm.resample("W").mean())
    assert np.allclose(movil_30.dropna().values, pm.rolling(30, min_periods=20).mean().dropna().values)
    print("✅ Ejercicio 2: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'pm.resample("W").mean(), pm.resample("MS").mean() y pm.rolling(30, min_periods=20).mean().')
    print("   Detalle:", e)

In [ ]:
# Las tres escalas en un gráfico
try:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(pm.index, pm.values, color="#c9c9d6", lw=0.6, label="diario")
    ax.plot(movil_30.index, movil_30.values, color="#4240e5", lw=1.6, label="media móvil 30 días")
    ax.plot(mensual.index, mensual.values, "o", color="#ff6b35", ms=3, label="promedio mensual")
    ax.axhline(50, color="crimson", ls="--", lw=1, label="norma 24 h (50 µg/m³)")
    ax.set_ylim(bottom=0); ax.set_ylabel("PM2.5 (µg/m³)"); ax.set_title(f"PM2.5 en {ESTACION}, 2020–2025")
    ax.legend(loc="upper right", fontsize=8); plt.tight_layout(); plt.show()
except Exception as e:
    print("Completa el ejercicio 2 para ver el gráfico.", e)

## 3. Estacionalidad: el año y la semana

¿Cuánto pesa el invierno? Promedia el PM2.5 por **mes del calendario** (todos los eneros juntos, todos los febreros juntos…). Luego haz lo mismo por **día de la semana** (`dayofweek`: 0 = lunes … 6 = domingo) y compara fines de semana con días hábiles.

### ✍️ Ejercicio 3 — Perfil mensual y semanal

In [ ]:
# TODO: promedio por mes del calendario (pm.index.month)
perfil_mes = ...
mes_peak = ...                  # TODO: el mes con el mayor promedio (idxmax)
razon_invierno_verano = ...     # TODO: máximo / mínimo del perfil mensual
# TODO: promedio sábado-domingo (dayofweek >= 5) menos promedio lunes-viernes
dif_fin_de_semana = ...
print("mes peak:", mes_peak, "| fin de semana - hábil:", dif_fin_de_semana)

In [ ]:
# ── Celda de chequeo · Ejercicio 3 ──────────────────────────────────────────
try:
    _p = pm.groupby(pm.index.month).mean()
    assert mes_peak == int(_p.idxmax())
    assert abs(razon_invierno_verano - _p.max() / _p.min()) < 1e-9
    assert abs(dif_fin_de_semana - (pm[pm.index.dayofweek >= 5].mean() - pm[pm.index.dayofweek < 5].mean())) < 1e-9
    print("✅ Ejercicio 3: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'pm.groupby(pm.index.month).mean(); .idxmax(); pm[pm.index.dayofweek >= 5].mean() - pm[pm.index.dayofweek < 5].mean().')
    print("   Detalle:", e)

## 4. Descomposición: tendencia + estacionalidad + resto

`seasonal_decompose` separa la serie **mensual** en sus tres ingredientes (con `period=12`: el ciclo se repite cada 12 meses). La **amplitud estacional** (máximo − mínimo del componente estacional) dice cuánto mueve el calendario; el cambio de la **tendencia** dice si el aire mejora o empeora de fondo.

### ✍️ Ejercicio 4 — Descompón la serie mensual

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose
# TODO: descompón `mensual` (aditivo, period=12)
desc = ...
amplitud_estacional = ...   # TODO: desc.seasonal.max() - desc.seasonal.min()
cambio_tendencia = ...      # TODO: último menos primer valor de desc.trend (sin NaN)
print("amplitud:", amplitud_estacional, "| cambio de tendencia:", cambio_tendencia)

In [ ]:
# ── Celda de chequeo · Ejercicio 4 ──────────────────────────────────────────
try:
    _d = seasonal_decompose(mensual, model="additive", period=12)
    assert abs(amplitud_estacional - (_d.seasonal.max() - _d.seasonal.min())) < 1e-6
    _t = _d.trend.dropna()
    assert abs(cambio_tendencia - (_t.iloc[-1] - _t.iloc[0])) < 1e-6
    print("✅ Ejercicio 4: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'seasonal_decompose(mensual, model="additive", period=12); desc.trend.dropna() y luego .iloc[-1] - .iloc[0].')
    print("   Detalle:", e)

In [ ]:
try:
    desc.plot(); plt.gcf().set_size_inches(9, 6); plt.tight_layout(); plt.show()
except Exception as e:
    print("Completa el ejercicio 4 para ver la descomposición.", e)

## 5. Validación temporal: tres pronósticos base

La regla de oro: **nunca evalúes un pronóstico con los datos que usaste para construirlo**. En series temporales eso significa **esconder el final**: entrenamos con los meses de 2020–2024 y pronosticamos los 12 meses de 2025 como si no los conociéramos. Nada de barajar.

Tres pronósticos base, de más tonto a menos:

- **Ingenuo:** todos los meses de 2025 = el último mes conocido (diciembre 2024).
- **Ingenuo estacional:** cada mes de 2025 = el mismo mes de 2024.
- **Promedio estacional:** cada mes de 2025 = el promedio de ese mes en 2020–2024.

Error: **MAE** (error absoluto medio, en µg/m³).

### ✍️ Ejercicio 5 — Pronostica 2025 con los tres métodos y mide el error

In [ ]:
entrena = mensual[:"2024"]          # 60 meses: 2020–2024
real_2025 = mensual["2025"]          # 12 meses escondidos
pron = {
    "ingenuo": pd.Series(entrena.iloc[-1], index=real_2025.index),
    # TODO: los 12 valores de 2024, con el índice de 2025
    "ingenuo_estacional": ...,
    # TODO: el promedio de cada mes del calendario en `entrena`, con el índice de 2025
    "promedio_estacional": ...,
}
# TODO: MAE de cada pronóstico contra real_2025 → diccionario {nombre: error}
errores = ...
print(errores)

In [ ]:
# ── Celda de chequeo · Ejercicio 5 ──────────────────────────────────────────
try:
    _e = mensual[:"2024"]; _r = mensual["2025"]
    _esp = {"ingenuo": abs(_e.iloc[-1] - _r).mean(),
            "ingenuo_estacional": np.abs(_e["2024"].values - _r.values).mean(),
            "promedio_estacional": np.abs(_e.groupby(_e.index.month).mean().values - _r.values).mean()}
    for _k, _v in _esp.items():
        assert abs(errores[_k] - _v) < 1e-6, f"el error de '{_k}' no calza"
    print("✅ Ejercicio 5: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'pd.Series(entrena["2024"].values, index=real_2025.index); entrena.groupby(entrena.index.month).mean().values; {k: float((p - real_2025).abs().mean()) for k, p in pron.items()}.')
    print("   Detalle:", e)

## 6. Un modelo simple: mes + tendencia

¿Le gana un modelo a los métodos base? Uno muy simple: una **regresión lineal** con 12 columnas 0/1 (una por mes) más una columna de **tiempo** (años desde enero de 2020). Captura la estacionalidad **y** la tendencia de fondo. La función que arma esas columnas ya está escrita; tú entrenas, pronosticas 2025 y eliges al ganador.

### ✍️ Ejercicio 6 — Entrena la regresión y elige el mejor método

In [ ]:
from sklearn.linear_model import LinearRegression

def X_mes_tendencia(idx):
    meses = np.eye(12)[idx.month - 1]                                    # 12 columnas 0/1
    t = ((idx.year - 2020) * 12 + idx.month - 1).to_numpy().reshape(-1, 1) / 12   # años desde ene-2020
    return np.hstack([meses, t])

# TODO: entrena LinearRegression(fit_intercept=False) con X_mes_tendencia(entrena.index) y entrena.values
reg = ...
# TODO: pronóstico de 2025 como Serie con el índice de real_2025, guárdalo en pron["regresion"]
# TODO: su MAE en errores["regresion"]
# TODO: el método con menor error
mejor_metodo = ...
print("mejor método:", mejor_metodo)

In [ ]:
# ── Celda de chequeo · Ejercicio 6 ──────────────────────────────────────────
try:
    _reg = LinearRegression(fit_intercept=False).fit(X_mes_tendencia(mensual[:"2024"].index), mensual[:"2024"].values)
    _p = _reg.predict(X_mes_tendencia(mensual["2025"].index))
    assert "regresion" in errores and abs(errores["regresion"] - np.abs(_p - mensual["2025"].values).mean()) < 1e-6
    assert mejor_metodo == min(errores, key=errores.get)
    print("✅ Ejercicio 6: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'reg = LinearRegression(fit_intercept=False).fit(X_mes_tendencia(entrena.index), entrena.values); pron["regresion"] = pd.Series(reg.predict(X_mes_tendencia(real_2025.index)), index=real_2025.index); mejor_metodo = min(errores, key=errores.get).')
    print("   Detalle:", e)

## 7. Prueba de fuego: lo que de verdad pasó en 2026

Elegiste el método con 2025. Ahora, como en la vida real, lo **reentrenas con todo lo que sabes** (2020–2025) y lo enfrentas a enero–septiembre de 2026, que el modelo nunca vio (`nuevos`). Compáralo con el ingenuo estacional (cada mes de 2026 = el mismo mes de 2025). ¿Se sostiene la elección?

### ✍️ Ejercicio 7 — Evalúa contra enero–septiembre de 2026

In [ ]:
# TODO: serie mensual 2026 de ESTACION (misma receta del ejercicio 1 + resample("MS").mean())
real_2026 = ...
# TODO: reentrena la regresión con TODO `mensual` (2020–2025) y pronostica los meses de real_2026
reg_total = ...
pron_2026 = ...
ingenuo_est_2026 = ...      # TODO: los primeros len(real_2026) meses de 2025, con el índice de 2026
mae_2026_regresion = ...
mae_2026_ingenuo_est = ...
print("MAE 2026 regresión:", mae_2026_regresion, "| ingenuo estacional:", mae_2026_ingenuo_est)

In [ ]:
# ── Celda de chequeo · Ejercicio 7 ──────────────────────────────────────────
try:
    _r26 = nuevos[nuevos["estacion"] == ESTACION].set_index("fecha")["pm25"].sort_index().asfreq("D").resample("MS").mean()
    assert len(real_2026) == len(_r26) == 9 and np.allclose(real_2026.values, _r26.values)
    _rg = LinearRegression(fit_intercept=False).fit(X_mes_tendencia(mensual.index), mensual.values)
    assert abs(mae_2026_regresion - np.abs(_rg.predict(X_mes_tendencia(_r26.index)) - _r26.values).mean()) < 1e-6
    assert abs(mae_2026_ingenuo_est - np.abs(mensual["2025"].values[:9] - _r26.values).mean()) < 1e-6
    print("✅ Ejercicio 7: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'Repite la receta del ejercicio 1 sobre `nuevos` y agrega .resample("MS").mean(); reentrena con X_mes_tendencia(mensual.index), mensual.values; ingenuo: mensual["2025"].values[:len(real_2026)].')
    print("   Detalle:", e)

In [ ]:
try:
    fig, ax = plt.subplots(figsize=(9, 3.5))
    ax.plot(mensual["2024":].index, mensual["2024":].values, color="#888", label="historia")
    ax.plot(real_2026.index, real_2026.values, "o-", color="black", label="real 2026")
    ax.plot(pron_2026.index, pron_2026.values, "s--", color="#4240e5", label="regresión")
    ax.plot(ingenuo_est_2026.index, ingenuo_est_2026.values, "^--", color="#ff6b35", label="ingenuo estacional")
    ax.set_ylim(bottom=0); ax.set_ylabel("PM2.5 mensual (µg/m³)"); ax.legend(fontsize=8)
    ax.set_title(f"{ESTACION}: pronósticos vs lo que pasó en 2026"); plt.tight_layout(); plt.show()
except Exception as e:
    print("Completa el ejercicio 7 para ver el gráfico.", e)

## ✍️ Reflexión (no se corrige, pero es lo más importante)

En 2025 ganó la regresión; en 2026, por poco, el ingenuo estacional. Julio de 2026 fue mucho
más limpio de lo habitual y **ningún** método lo anticipó. Si tu jefatura te pide «el pronóstico
de PM2.5 para el próximo invierno», ¿qué le entregas: un número, un rango, o dos métodos con su
historial de error? Escríbelo en dos frases.

## Cierre

- Una serie temporal necesita **calendario completo** (`asfreq`): los días sin dato se ven, no se saltan.
- **Remuestrear** y **media móvil** cambian la escala para ver la forma; la **descomposición**
  separa tendencia, estacionalidad y resto.
- En el PM2.5 de Cerro Navia, el **invierno** manda (el peor mes promedia casi 5 veces el mejor);
  el ciclo **semanal** existe pero es chico en comparación.
- **Validación temporal:** se esconde el final, nunca se baraja. El **ingenuo estacional** es la
  vara mínima: en esta serie, ganarle no es fácil.
- Un método que gana un año puede perder el siguiente: reporta el **error histórico**, no solo el número.

> *En la profundización aplicarás lo mismo a los **sismos**: ¿tienen temporada? La respuesta
> honesta es otra, y aprenderás a no dejarte engañar por un test mal elegido.*